# Groq Frontier RAGAS-style Evaluation

Evaluates `llama-3.3-70b-versatile` and `qwen/qwen3-32b` using the same
custom metric implementation as `qlora_ragas_eval.ipynb`.

No RAGAS `evaluate()` call — metrics implemented directly via Groq API.

**Run after** `qlora_ragas_eval.ipynb` — loads QLoRA results from CSV
and produces a combined comparison chart.

## 0 · Install

In [ ]:
%pip install -q groq sentence-transformers scikit-learn

## 1 · Config

In [ ]:
import os
from pathlib import Path

os.environ["GROQ_API_KEY"] = "gsk_..."          # ← your Groq key
PINECONE_KEY               = "YOUR_PINECONE_KEY" # ← your Pinecone key

# Judge model — used for faithfulness, context_precision, context_recall
# Uses a SEPARATE model from the generator so judge is always neutral
JUDGE_MODEL = "llama-3.1-8b-instant"   # separate daily quota from 70b

# Models to evaluate as generators
GEN_MODELS = [
    "llama-3.3-70b-versatile",
    "qwen/qwen3-32b",
]

TOP_K          = 10
RETRIEVER_MODE = "advanced"
QUICK_TEST     = True
QUICK_N        = 5      # keep small — each record uses ~15 judge calls
GEN_DELAY      = 2.5    # seconds between generation calls
JUDGE_DELAY    = 4.0    # seconds between judge calls (prevents rate limit)

DATA_DIR = Path(".")
print("✅ Config set")
print(f"   Judge model     : {JUDGE_MODEL}")
print(f"   Generator models: {GEN_MODELS}")
print(f"   Records per model: {QUICK_N}")

## 2 · Metric implementations

In [ ]:
import time, re
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

_judge_client = Groq(api_key=os.environ["GROQ_API_KEY"])
_sem = SentenceTransformer(
    "flax-sentence-embeddings/st-codesearch-distilroberta-base"
)


def _judge(prompt: str) -> str:
    time.sleep(JUDGE_DELAY)
    resp = _judge_client.chat.completions.create(
        model       = JUDGE_MODEL,
        messages    = [{"role": "user", "content": prompt}],
        max_tokens  = 256,
        temperature = 0,
    )
    return resp.choices[0].message.content.strip()


def faithfulness(answer: str, contexts: list) -> float:
    ctx = "\n---\n".join(contexts[:5])
    # Step 1: extract claims
    claims_raw = _judge(
        f"Given this code answer, list every factual claim it makes "
        f"(one per line). If it is pure code, describe what it does in "
        f"1-3 short statements.\n\nAnswer:\n{answer}\n\nClaims:"
    )
    claims = [c.strip().lstrip("•-0123456789. ")
              for c in claims_raw.splitlines() if c.strip()]
    if not claims:
        return float("nan")
    # Step 2: verify each claim
    supported = sum(
        1 for claim in claims
        if "YES" in _judge(
            f"Context:\n{ctx}\n\nClaim: {claim}\n\n"
            f"Is this claim supported by the context? Answer YES or NO only."
        ).upper()
    )
    return supported / len(claims)


def answer_relevancy(question: str, answer: str) -> float:
    e = _sem.encode([question, answer])
    return float(cosine_similarity([e[0]], [e[1]])[0][0])


def context_precision(question: str, contexts: list, answer: str) -> float:
    scores = [
        1.0 if "YES" in _judge(
            f"Question: {question}\n\nChunk:\n{ctx}\n\n"
            f"Is this chunk useful for answering the question? YES or NO only."
        ).upper() else 0.0
        for ctx in contexts[:5]
    ]
    return sum(scores) / len(scores) if scores else float("nan")


def context_recall(contexts: list, ground_truth: str) -> float:
    ctx = "\n---\n".join(contexts[:5])
    verdict = _judge(
        f"Context:\n{ctx}\n\nGround truth:\n{ground_truth}\n\n"
        f"Can the ground truth be derived from the context? YES or NO only."
    ).upper()
    return 1.0 if "YES" in verdict else 0.0


print(f"✅ Metrics ready  (judge: {JUDGE_MODEL})")

## 3 · Load data and retriever

In [ ]:
import json, random
import rag_pipeline, importlib
importlib.reload(rag_pipeline)
from rag_pipeline import VersionControlRAG

# ingest_only=True — embedder on CPU, no Qwen loaded, GPU stays free
rag = VersionControlRAG(
    pinecone_key = PINECONE_KEY,
    model_path   = "Qwen/Qwen2.5-Coder-7B",
    adapter_path = None,
    ingest_only  = True,
)
rag.load_local_corpus(str(DATA_DIR / "local_corpus.json"))

with open(DATA_DIR / "ragas_dataset.jsonl") as f:
    all_records = [json.loads(l) for l in f]
with open(DATA_DIR / "fastapi_golden_set_splits.json") as f:
    splits = json.load(f)

test_q       = {x["instruction"] for x in splits["test"]}
test_records = [r for r in all_records if r["question"] in test_q]
random.seed(42)
eval_records = random.sample(test_records, QUICK_N) if QUICK_TEST else test_records

print(f"Corpus        : {len(rag.local_corpus)} docs")
print(f"Eval records  : {len(eval_records)}")

## 4 · Generator function

In [ ]:
from groq import Groq

SYSTEM_PROMPT = (
    "You are a FastAPI expert. Use the provided code context to answer "
    "the question. Start with a single Python comment (# ...) describing "
    "what the code does, then write the Python code. No markdown fences."
)

_gen_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate_groq(model_id: str, question: str, contexts: list) -> str:
    ctx = "\n\n".join(contexts[:5])
    time.sleep(GEN_DELAY)
    resp = _gen_client.chat.completions.create(
        model    = model_id,
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content":
             f"### Context\n{ctx}\n\n### Question\n{question}\n\n### Answer"},
        ],
        max_tokens=512, temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = "\n".join(l for l in raw.splitlines()
                        if not l.strip().startswith("```")).strip()
    return raw or "# model returned empty response"


print("✅ Generator ready")

## 5 · Run evaluation

Runs each model in sequence.  
60s pause between models so the daily token quota has time to breathe.

In [ ]:
import pandas as pd
from tqdm import tqdm

METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]


def run_one_model(model_id: str, records: list) -> pd.DataFrame:
    print(f"\n{'='*55}")
    print(f"  {model_id}  ({len(records)} records)")
    print(f"{'='*55}")

    rows, gen_errors = [], 0

    for i, r in enumerate(tqdm(records, desc=model_id[:35])):
        # Retrieve
        try:
            contexts = rag.retrieve_complex(
                r["question"], top_k=TOP_K, mode=RETRIEVER_MODE
            ) or r["contexts"][:TOP_K]
        except Exception as e:
            print(f"  Retrieval [{i}]: {e}")
            contexts = r["contexts"][:TOP_K]

        # Generate
        try:
            answer = generate_groq(model_id, r["question"], contexts)
        except Exception as e:
            print(f"  Generation [{i}]: {type(e).__name__}: {e}")
            answer = "# generation error"
            gen_errors += 1

        row = {"question": r["question"], "model": model_id}

        # answer_relevancy — local embeddings, no API call
        row["answer_relevancy"] = answer_relevancy(r["question"], answer)

        # faithfulness
        try:
            row["faithfulness"] = faithfulness(answer, contexts)
        except Exception as e:
            print(f"  faithfulness [{i}]: {type(e).__name__}: {e}")
            row["faithfulness"] = float("nan")

        # context_precision
        try:
            row["context_precision"] = context_precision(
                r["question"], contexts, answer
            )
        except Exception as e:
            print(f"  context_precision [{i}]: {type(e).__name__}: {e}")
            row["context_precision"] = float("nan")

        # context_recall
        try:
            row["context_recall"] = context_recall(
                contexts, r["ground_truth"]
            )
        except Exception as e:
            print(f"  context_recall [{i}]: {type(e).__name__}: {e}")
            row["context_recall"] = float("nan")

        rows.append(row)
        print(f"  [{i+1}/{len(records)}] "
              f"faith={row['faithfulness']:.2f}  "
              f"rel={row['answer_relevancy']:.2f}  "
              f"prec={row['context_precision']:.2f}  "
              f"rec={row['context_recall']:.2f}")

    if gen_errors:
        print(f"  ⚠ {gen_errors} generation errors")

    df = pd.DataFrame(rows)
    print(f"\n  Mean scores:")
    for col in METRIC_COLS:
        if col in df.columns:
            val = df[col].mean()
            print(f"    {col:25s}: {val:.3f}" if pd.notna(val)
                  else f"    {col:25s}: NaN")
    return df


print("✅ run_one_model defined")

In [ ]:
# Run all generator models
groq_dfs = []

for i, model_id in enumerate(GEN_MODELS):
    df = run_one_model(model_id, eval_records)
    groq_dfs.append(df)

    if i < len(GEN_MODELS) - 1:
        print(f"\nPausing 60s between models (token quota)...\n")
        time.sleep(60)

df_groq = pd.concat(groq_dfs, ignore_index=True)
print("\nAll Groq models done.")

## 6 · Combined comparison with QLoRA results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load QLoRA results if available
qlora_csv = DATA_DIR / "model_comparison_results.csv"
if qlora_csv.exists():
    df_qlora = pd.read_csv(qlora_csv)
    df_qlora = df_qlora[df_qlora["model"] == "qwen_qlora"].copy()
    df_all   = pd.concat([df_qlora, df_groq], ignore_index=True)
    print(f"Loaded QLoRA results ({len(df_qlora)} rows)")
else:
    df_all = df_groq
    print("QLoRA results not found — run qlora_ragas_eval.ipynb first")

exist   = [c for c in METRIC_COLS if c in df_all.columns]
summary = (
    df_all.groupby("model")[exist]
    .mean().round(3)
    .sort_values("faithfulness", ascending=False, na_position="last")
)

print("\n" + "="*70)
print(f"{'Model':30s}" + "".join(f"  {c[:10]:>10}" for c in exist))
print("-"*70)
for model, row in summary.iterrows():
    vals = "".join(
        f"  {row[c]:10.3f}" if pd.notna(row[c]) else f"  {'NaN':>10}"
        for c in exist
    )
    tag = "  ← your model" if model == "qwen_qlora" else ""
    print(f"{model:30s}{vals}{tag}")
print("="*70)

In [ ]:
# Bar chart
model_order = list(summary.index)
palette = {
    "qwen_qlora":               "#7F77DD",
    "llama-3.3-70b-versatile":  "#1D9E75",
    "qwen/qwen3-32b":           "#D85A30",
}
colors = [palette.get(m, "#B4B2A9") for m in model_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All metrics bar chart
summary.reindex(model_order).T.plot.bar(
    ax=axes[0], color=colors, width=0.6
)
axes[0].set_title("RAGAS-style scores: QLoRA vs Frontier")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(exist, rotation=20, ha="right")
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].legend(title="Model", fontsize=8)

# Faithfulness only — most important metric
if "faithfulness" in summary.columns:
    faith_vals = summary.reindex(model_order)["faithfulness"]
    bars = axes[1].bar(
        range(len(model_order)),
        faith_vals.values,
        color=colors, width=0.5, edgecolor="white"
    )
    axes[1].set_xticks(range(len(model_order)))
    axes[1].set_xticklabels(
        [m.split("/")[-1][:20] for m in model_order],
        rotation=20, ha="right", fontsize=9
    )
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Faithfulness (primary metric)")
    axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    for bar, val in zip(bars, faith_vals.values):
        if pd.notna(val):
            axes[1].text(
                bar.get_x() + bar.get_width()/2, val + 0.02,
                f"{val:.3f}", ha="center", fontsize=9
            )

plt.tight_layout()
plt.savefig(DATA_DIR / "model_comparison_chart.png", dpi=150)
plt.show()
print("Saved: model_comparison_chart.png")

## 7 · Save

In [ ]:
df_groq.to_csv(DATA_DIR / "groq_results.csv",         index=False)
df_all.to_csv( DATA_DIR / "full_comparison.csv",       index=False)
summary.to_csv(DATA_DIR / "full_comparison_summary.csv")

print("Saved:")
print("  groq_results.csv             — Groq model results only")
print("  full_comparison.csv          — all models combined")
print("  full_comparison_summary.csv  — mean scores per model")
print()
print(summary.to_string())